# CLSA RETFound retinal-age algorithm fairness audit

## Quick use

1. Attach a Databricks GPU cluster with enough driver memory for the
   participant-level embedding matrix. If coverage reconciliation proves every
   passing image already has a vector, the GPU is not used.
2. Confirm the hard-coded repository and Volume paths in **Configuration**.
3. Run all cells. Completed embedding-rollup batches, the age model, and the OOF
   predictions are reused automatically after a restart.
4. Review `16_algorithm_fairness/04_statistics`, `05_figures`, and the generated
   `RUN_README.md`. Participant identifiers are confined to `01_private`.

This notebook audits the complete CLSA fundus manifest: technical-quality
selection, frozen RETFound representations, and a single CLSA-wide retinal-age
head. Missing quality/vector batches are completed and checkpointed. The primary
model contains no race/demographic predictors and is evaluated
only with participant-grouped out-of-fold (OOF) predictions. Images/eyes are
pooled within visit and one visit (baseline preferred) is retained per participant.

The primary race comparison matches each sufficiently large self-reported
cultural/racial-background group separately to White participants on age
(±1 year), sex, and released comorbidities. A ±2-year sensitivity analysis is
also saved. “Black” is the released self-reported category; it is **not** treated
as African genetic ancestry. Multiple selections remain `Multiple groups`.

Performance differences do not by themselves prove that RETFound is inherently
biased. They estimate disparities in this full pipeline within CLSA and may also
reflect image acquisition, quality selection, sample composition, outcome
measurement, or limited overlap. Small cells are suppressed from figures and
excluded from inferential matching.


In [ ]:
%pip install -q "pandas>=2.1,<3" "pyarrow>=14" "scikit-learn>=1.4,<2" "scipy>=1.11,<2" "statsmodels~=0.14.2" "seaborn>=0.13,<1" "joblib>=1.3"


In [ ]:
dbutils.library.restartPython()


In [ ]:
dbutils.widgets.text("hf_token", "", "Hugging Face token (only if missing vectors require download)")


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import shutil
import sys
import time
import uuid
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from pyspark.sql import functions as F
from statsmodels.stats.proportion import proportion_confint

sns.set_theme(style="whitegrid", context="talk")


## Configuration


In [ ]:
repo_root = Path(
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina"
)
dataset_root = Path(
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset"
)
derived_root = dataset_root / "derived" / "clsa_retinal_aging"
retfound_root = derived_root / "fundus_retfound"
full_image_manifest_path = derived_root / "fundus_image_manifest"
quality_path = retfound_root / "01_quality" / "fundus_quality_manifest.parquet"
embedding_root = retfound_root / "02_embeddings"
sap_questionnaire_path = derived_root / "sap_questionnaire_visit"
baseline_archive_path = dataset_root / "2209017_UOttawa_EFreeman_BL.zip"
baseline_member_suffix = "CoPv7_Qx_CANUE_PA_BS.csv"

output_root = derived_root / "Age_Glaucoma" / "16_algorithm_fairness"
full_pipeline_root = output_root / "00_full_image_pipeline"
private_root = output_root / "01_private"
rollup_root = output_root / "02_embedding_rollup_batches"
model_root = output_root / "03_age_model"
statistics_root = output_root / "04_statistics"
figure_root = output_root / "05_figures"
for directory in (
    full_pipeline_root, private_root, rollup_root, model_root, statistics_root, figure_root
):
    directory.mkdir(parents=True, exist_ok=True)

# Prespecified analysis parameters.
expected_embedding_dim = 1024
ridge_alpha = 10.0
cv_folds = 5
primary_age_caliper_years = 1.0
sensitivity_age_caliper_years = 2.0
match_ratio = 2
minimum_inference_group_n = 30
minimum_reporting_group_n = 10
bootstrap_repetitions = 1000
random_state = 20260821
resume_completed_outputs = True
process_missing_full_manifest_images = True
retry_prior_embedding_failures = True
image_pipeline_batch_size = 500
retfound_gpu_batch_size = 8
retfound_repo = None
checkpoint_path = None
allow_model_downloads = True

if minimum_reporting_group_n < 5:
    raise ValueError("minimum_reporting_group_n must be at least 5")
if minimum_inference_group_n < minimum_reporting_group_n:
    raise ValueError("Inference threshold cannot be below reporting threshold")

module_root = repo_root / "src"
if not module_root.exists():
    raise FileNotFoundError(f"Repository source directory not found: {module_root}")
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

import retfound_fairness as _fairness
_fairness = importlib.reload(_fairness)
from fundus_retfound_pipeline import (
    AgeModelConfig,
    QualityConfig,
    RETFoundConfig,
    extract_retfound_embeddings,
    load_retfound_model,
    run_quality_pipeline,
    train_age_head,
    write_frame,
    write_json,
)
from retfound_fairness import (
    benjamini_hochberg,
    classify_racial_background,
    fairness_metric_table,
    match_group_to_reference,
    matched_outcome_contrasts,
    pool_age_predictions_to_participants,
    standardized_mean_differences,
)

print("Fairness helper:", _fairness.__file__)
print("Output root:", output_root)


## 1. Reconcile the complete image manifest with durable quality/vector outputs

The Delta image manifest from notebook 01 is the denominator—not whichever
embedding batches happen to exist. Completed notebook 02 artifacts are reused,
but any manifest image absent from quality processing is run through the same
CLSA quality pipeline in resumable 500-image batches. Any quality-passing image
without either a vector or a recorded embedding failure is vectorized in a
separate resumable batch. Thus every raw manifest image is explicitly accounted
for as quality-pass, quality-fail, embedded, or embedding-failed.


In [ ]:
required_inputs = {
    "full fundus image manifest": full_image_manifest_path,
    "SAP questionnaire Delta table": sap_questionnaire_path,
    "baseline questionnaire ZIP": baseline_archive_path,
}
missing = [f"{label}: {path}" for label, path in required_inputs.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Required completed inputs are missing:\n- " + "\n- ".join(missing))

existing_embedding_batch_paths = sorted(
    (embedding_root / "batches").glob("batch_*/retfound_embeddings.parquet")
)
consolidated_embedding_path = embedding_root / "retfound_embeddings.parquet"
if existing_embedding_batch_paths:
    existing_embedding_paths = existing_embedding_batch_paths
    embedding_source_mode = "completed_notebook02_batches"
elif consolidated_embedding_path.exists():
    existing_embedding_paths = [consolidated_embedding_path]
    embedding_source_mode = "consolidated_notebook02"
else:
    existing_embedding_paths = []
    embedding_source_mode = "no_prior_embeddings"

print("Prior embedding source mode:", embedding_source_mode)
print("Prior embedding input files:", len(existing_embedding_paths))


In [ ]:
def normalize_visit(series):
    return series.astype("string").str.upper().replace({"FUP1": "F1"})


def normalize_identifier(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)


def truthy_quality(series):
    return series.astype("string").str.upper().isin(["TRUE", "1", "1.0", "Y", "YES"])


def publish_local_file(local_path, destination, attempts=4):
    # Publish one durable artifact through an atomic temporary Volume path.
    local_path = Path(local_path)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, attempts + 1):
        temporary = destination.with_name(destination.name + f".partial-{uuid.uuid4().hex}")
        try:
            shutil.copyfile(local_path, temporary)
            if temporary.stat().st_size != local_path.stat().st_size:
                raise OSError("Volume copy size mismatch")
            os.replace(temporary, destination)
            return destination
        except Exception:
            if temporary.exists():
                temporary.unlink()
            if attempt == attempts:
                raise
            time.sleep(2 ** (attempt - 1))


def normalize_image_frame(frame):
    frame = frame.copy()
    participant_column = next(
        (column for column in ("participant_id", "participant_id_parsed") if column in frame.columns),
        None,
    )
    if participant_column is None:
        raise ValueError("Image table lacks participant_id/participant_id_parsed")
    if participant_column != "participant_id":
        frame = frame.rename(columns={participant_column: "participant_id"})
    frame["participant_id"] = normalize_identifier(frame["participant_id"])
    frame["visit"] = normalize_visit(frame["visit"])
    frame["image_path"] = frame["image_path"].astype(str)
    return frame.dropna(subset=["participant_id", "visit", "image_path"])


manifest_spark = spark.read.format("delta").load(str(full_image_manifest_path))
manifest_participant_column = next(
    (column for column in ("participant_id", "participant_id_parsed") if column in manifest_spark.columns),
    None,
)
if manifest_participant_column is None or "image_path" not in manifest_spark.columns:
    raise ValueError(
        "Full image manifest must contain image_path and participant_id_parsed/participant_id"
    )
manifest_selections = [
    F.col(manifest_participant_column).cast("string").alias("participant_id"),
    F.col("image_path").cast("string").alias("image_path"),
    F.col("visit").cast("string").alias("visit"),
]
for source_column, output_column in (
    ("eye_parsed", "eye"),
    ("filename", "filename"),
    ("relative_path", "relative_path"),
):
    if source_column in manifest_spark.columns:
        manifest_selections.append(
            F.col(source_column).cast("string").alias(output_column)
        )
full_manifest = normalize_image_frame(
    manifest_spark.select(*manifest_selections).dropDuplicates(["image_path"]).toPandas()
).sort_values(["participant_id", "visit", "image_path"], kind="stable").reset_index(drop=True)
full_manifest_paths = set(full_manifest["image_path"])
if not full_manifest_paths:
    raise ValueError("The full fundus image manifest contains no usable image paths")

quality_frames = []
if quality_path.exists():
    quality_frames.append(pd.read_parquet(quality_path))
fairness_quality_batch_root = full_pipeline_root / "01_quality_batches"
fairness_quality_batch_root.mkdir(parents=True, exist_ok=True)
for cached_path in sorted(
    fairness_quality_batch_root.glob("batch_*/fundus_quality_manifest.parquet")
):
    quality_frames.append(pd.read_parquet(cached_path))
quality = (
    normalize_image_frame(pd.concat(quality_frames, ignore_index=True))
    if quality_frames
    else pd.DataFrame(columns=["participant_id", "visit", "image_path", "quality_pass"])
)
if not quality.empty:
    quality = quality.drop_duplicates("image_path", keep="last").reset_index(drop=True)

missing_quality_paths = sorted(full_manifest_paths - set(quality.get("image_path", [])))
print(
    f"Full manifest denominator: {len(full_manifest):,} images from "
    f"{full_manifest['participant_id'].nunique():,} unique participants"
)
print(f"Images missing technical-quality processing: {len(missing_quality_paths):,}")
if missing_quality_paths and not process_missing_full_manifest_images:
    raise RuntimeError(
        "Not all fundus images entered quality processing. Set "
        "process_missing_full_manifest_images=True or complete notebook 02."
    )

quality_config = QualityConfig(
    output_size=256,
    model_input_size=224,
    save_preprocessed=False,
)
missing_quality_manifest = full_manifest[
    full_manifest["image_path"].isin(missing_quality_paths)
].reset_index(drop=True)
new_quality_frames = []
for start in range(0, len(missing_quality_manifest), image_pipeline_batch_size):
    stop = min(start + image_pipeline_batch_size, len(missing_quality_manifest))
    batch = missing_quality_manifest.iloc[start:stop].copy()
    digest = hashlib.sha1(
        "\n".join(batch["image_path"]).encode("utf-8")
    ).hexdigest()[:12]
    batch_root = fairness_quality_batch_root / f"batch_{digest}"
    batch_cache = batch_root / "fundus_quality_manifest.parquet"
    if resume_completed_outputs and batch_cache.exists():
        completed = pd.read_parquet(batch_cache)
    else:
        local_root = Path("/local_disk0/tmp") / f"fairness-quality-{digest}"
        completed = run_quality_pipeline(batch, local_root, quality_config)
        batch_root.mkdir(parents=True, exist_ok=True)
        publish_local_file(
            local_root / "fundus_quality_manifest.parquet",
            batch_cache,
        )
        shutil.rmtree(local_root, ignore_errors=True)
    new_quality_frames.append(completed)
    print(
        f"[full quality {stop:,}/{len(missing_quality_manifest):,}] "
        f"accounted for {len(completed):,} images",
        flush=True,
    )

if new_quality_frames:
    quality = normalize_image_frame(
        pd.concat([quality, *new_quality_frames], ignore_index=True)
    )
quality = quality.drop_duplicates("image_path", keep="last").reset_index(drop=True)
quality = quality[quality["image_path"].isin(full_manifest_paths)].reset_index(drop=True)
quality["quality_pass_bool"] = truthy_quality(quality["quality_pass"])
quality_paths = set(quality["image_path"])
missing_after_quality = full_manifest_paths - quality_paths
if missing_after_quality:
    raise RuntimeError(
        f"Quality coverage remains incomplete for {len(missing_after_quality):,} manifest images"
    )
quality_pass_paths = set(
    quality.loc[quality["quality_pass_bool"], "image_path"].astype(str)
)
print(
    f"Complete quality ledger: {len(quality):,}/{len(full_manifest):,} manifest images; "
    f"{len(quality_pass_paths):,} passed and "
    f"{len(full_manifest_paths - quality_pass_paths):,} failed"
)


In [ ]:
def embedding_paths_and_failures(paths):
    embedded = set()
    failures = set()
    usable_paths = []
    for path in paths:
        try:
            identifiers = pd.read_parquet(path, columns=["image_path"])
        except Exception as exc:
            raise RuntimeError(f"Unable to read embedding ledger {path}: {exc}") from exc
        embedded.update(identifiers["image_path"].dropna().astype(str))
        usable_paths.append(path)
        failure_path = path.parent / "retfound_embedding_failures.csv"
        if failure_path.exists() and failure_path.stat().st_size:
            try:
                failure_frame = pd.read_csv(failure_path)
            except pd.errors.EmptyDataError:
                failure_frame = pd.DataFrame()
            if "image_path" in failure_frame.columns:
                failures.update(failure_frame["image_path"].dropna().astype(str))
    return usable_paths, embedded, failures


fairness_embedding_batch_root = full_pipeline_root / "02_embedding_batches"
fairness_embedding_batch_root.mkdir(parents=True, exist_ok=True)
cached_fairness_embedding_paths = sorted(
    fairness_embedding_batch_root.glob("batch_*/retfound_embeddings.parquet")
)
embedding_paths, embedded_paths, embedding_failure_paths = embedding_paths_and_failures(
    [*existing_embedding_paths, *cached_fairness_embedding_paths]
)
# Notebook 02 also writes a consolidated failure ledger.
consolidated_failure_path = embedding_root / "retfound_embedding_failures.csv"
if consolidated_failure_path.exists() and consolidated_failure_path.stat().st_size:
    try:
        consolidated_failures = pd.read_csv(consolidated_failure_path)
    except pd.errors.EmptyDataError:
        consolidated_failures = pd.DataFrame()
    if "image_path" in consolidated_failures.columns:
        embedding_failure_paths.update(
            consolidated_failures["image_path"].dropna().astype(str)
        )

missing_embedding_paths = sorted(
    quality_pass_paths - embedded_paths
    if retry_prior_embedding_failures
    else quality_pass_paths - embedded_paths - embedding_failure_paths
)
print(f"Quality-passing images with prior vectors: {len(embedded_paths & quality_pass_paths):,}")
print(f"Quality-passing images with recorded embedding failure: {len(embedding_failure_paths & quality_pass_paths):,}")
print(
    "Quality-passing images requiring RETFound now "
    f"(including prior failures={retry_prior_embedding_failures}): "
    f"{len(missing_embedding_paths):,}"
)
if missing_embedding_paths and not process_missing_full_manifest_images:
    raise RuntimeError(
        "Not all quality-passing fundus images were attempted by RETFound. Set "
        "process_missing_full_manifest_images=True."
    )

if missing_embedding_paths:
    if not torch.cuda.is_available():
        raise RuntimeError(
            f"{len(missing_embedding_paths):,} quality-passing images still need vectors. "
            "Attach this notebook to GPU compute, then rerun; completed quality batches "
            "will resume."
        )
    retfound_config = RETFoundConfig(
        repo_path=retfound_repo,
        checkpoint_path=checkpoint_path,
        allow_downloads=allow_model_downloads,
        device="cuda",
        batch_size=retfound_gpu_batch_size,
    )
    temporary_hf_token = dbutils.widgets.get("hf_token").strip()
    if temporary_hf_token:
        os.environ["HF_TOKEN"] = temporary_hf_token
    try:
        retfound_model, retfound_device, resolved_repo, resolved_checkpoint = (
            load_retfound_model(retfound_config)
        )
    finally:
        os.environ.pop("HF_TOKEN", None)
        temporary_hf_token = ""
    print("Completing missing vectors on:", retfound_device)
    print("RETFound repository:", resolved_repo)
    print("RETFound checkpoint:", resolved_checkpoint)

    quality_for_embedding = quality[
        quality["image_path"].isin(missing_embedding_paths)
    ].sort_values("image_path", kind="stable").reset_index(drop=True)
    for start in range(0, len(quality_for_embedding), image_pipeline_batch_size):
        stop = min(start + image_pipeline_batch_size, len(quality_for_embedding))
        batch = quality_for_embedding.iloc[start:stop].copy()
        digest = hashlib.sha1(
            "\n".join(batch["image_path"]).encode("utf-8")
        ).hexdigest()[:12]
        batch_root = fairness_embedding_batch_root / f"batch_{digest}"
        batch_cache = batch_root / "retfound_embeddings.parquet"
        if not (resume_completed_outputs and batch_cache.exists()):
            local_root = Path("/local_disk0/tmp") / f"fairness-embedding-{digest}"
            all_failed = False
            try:
                extract_retfound_embeddings(
                    batch,
                    local_root,
                    retfound_config,
                    quality_config,
                    model=retfound_model,
                    device=retfound_device,
                    checkpoint_path=resolved_checkpoint,
                    force=True,
                )
            except RuntimeError as exc:
                local_failure = local_root / "retfound_embedding_failures.csv"
                if "Every image failed" not in str(exc) or not local_failure.exists():
                    raise
                all_failed = True
            batch_root.mkdir(parents=True, exist_ok=True)
            local_failure = local_root / "retfound_embedding_failures.csv"
            if not all_failed:
                publish_local_file(local_root / "retfound_embeddings.parquet", batch_cache)
            if local_failure.exists():
                publish_local_file(
                    local_failure,
                    batch_root / "retfound_embedding_failures.csv",
                )
            shutil.rmtree(local_root, ignore_errors=True)
        if batch_cache.exists():
            embedding_paths.append(batch_cache)
        print(
            f"[full RETFound {stop:,}/{len(quality_for_embedding):,}] "
            f"saved/resumed {batch_cache}",
            flush=True,
        )
    del retfound_model
    torch.cuda.empty_cache()

embedding_paths, embedded_paths, embedding_failure_paths = embedding_paths_and_failures(
    sorted(set(embedding_paths))
)
for failure_path in sorted(
    fairness_embedding_batch_root.glob("batch_*/retfound_embedding_failures.csv")
):
    try:
        failure_frame = pd.read_csv(failure_path)
    except pd.errors.EmptyDataError:
        continue
    if "image_path" in failure_frame.columns:
        embedding_failure_paths.update(
            failure_frame["image_path"].dropna().astype(str)
        )
unresolved_embedding_failure_paths = embedding_failure_paths - embedded_paths
unaccounted_quality_pass = (
    quality_pass_paths - embedded_paths - unresolved_embedding_failure_paths
)
if unaccounted_quality_pass:
    raise RuntimeError(
        f"RETFound coverage remains incomplete for {len(unaccounted_quality_pass):,} "
        "quality-passing manifest images"
    )
if not embedded_paths:
    raise RuntimeError("No completed RETFound vectors are available")

image_coverage = pd.DataFrame(
    [
        {"stage": "Full image manifest", "n_images": len(full_manifest_paths)},
        {"stage": "Technical-quality attempted", "n_images": len(quality_paths)},
        {"stage": "Technical-quality passed", "n_images": len(quality_pass_paths)},
        {"stage": "RETFound vector completed", "n_images": len(embedded_paths & quality_pass_paths)},
        {"stage": "RETFound attempt failed", "n_images": len(unresolved_embedding_failure_paths & quality_pass_paths)},
    ]
)
write_frame(image_coverage, statistics_root / "full_image_coverage.parquet")
display(image_coverage)
print("Embedding Parquets used:", len(embedding_paths))


## 2. Resumable participant-visit embedding rollup

Each source batch is reduced to participant-visit mean embeddings and saved as
an interim Parquet. A second streaming pass combines participants that crossed
source-batch boundaries. This avoids loading all image-level 1024-vectors into
the driver and can resume after a cluster failure.


In [ ]:
def rollup_embedding_batch(source_path, destination_path):
    frame = pd.read_parquet(source_path)
    frame.attrs = {}
    required = {"image_path", "participant_id", "visit", "embedding"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{source_path} missing columns: {sorted(missing)}")
    frame["participant_id"] = normalize_identifier(frame["participant_id"])
    frame["visit"] = normalize_visit(frame["visit"])
    if "quality_pass" in frame.columns:
        frame = frame[truthy_quality(frame["quality_pass"])].copy()
    frame = frame[frame["image_path"].astype(str).isin(quality_pass_paths)].copy()
    rows = []
    for (participant_id, visit), subset in frame.groupby(["participant_id", "visit"], dropna=False):
        vectors = [np.asarray(value, dtype=np.float32).reshape(-1) for value in subset["embedding"]]
        if not vectors:
            continue
        dimensions = {vector.size for vector in vectors}
        if dimensions != {expected_embedding_dim}:
            raise ValueError(
                f"Invalid embedding dimensions in {source_path}: {sorted(dimensions)}"
            )
        rows.append(
            {
                "participant_id": str(participant_id),
                "visit": str(visit),
                "embedding_sum": np.stack(vectors).sum(axis=0, dtype=np.float64),
                "n_embedded_images": len(vectors),
            }
        )
    local = Path("/local_disk0/tmp") / f"fairness-rollup-{uuid.uuid4().hex}.parquet"
    write_frame(pd.DataFrame(rows), local)
    publish_local_file(local, destination_path)
    local.unlink(missing_ok=True)
    return len(rows), len(frame)


rollup_paths = []
for index, source_path in enumerate(embedding_paths, 1):
    source_label = (
        f"{source_path.parent.name}_"
        f"{hashlib.sha1(str(source_path).encode('utf-8')).hexdigest()[:8]}"
    )
    destination = rollup_root / f"{source_label}_participant_visit_rollup.parquet"
    if resume_completed_outputs and destination.exists():
        rollup_paths.append(destination)
        if index == 1 or index % 25 == 0 or index == len(embedding_paths):
            print(f"[rollup {index}/{len(embedding_paths)}] resumed {destination.name}")
        continue
    n_rollups, n_images = rollup_embedding_batch(source_path, destination)
    rollup_paths.append(destination)
    print(
        f"[rollup {index}/{len(embedding_paths)}] {n_images:,} images -> "
        f"{n_rollups:,} participant-visits; saved {destination.name}"
    )


In [ ]:
participant_visit_embedding_path = private_root / "participant_visit_embeddings.parquet"
participant_visit_embedding_metadata_path = (
    private_root / "participant_visit_embeddings_metadata.json"
)
rollup_signature_text = "\n".join(
    f"{path}|{path.stat().st_size}|{path.stat().st_mtime_ns}"
    for path in sorted(rollup_paths)
)
rollup_signature = hashlib.sha256(rollup_signature_text.encode("utf-8")).hexdigest()
resume_final_rollup = False
if (
    resume_completed_outputs
    and participant_visit_embedding_path.exists()
    and participant_visit_embedding_metadata_path.exists()
):
    prior_rollup_metadata = json.loads(
        participant_visit_embedding_metadata_path.read_text()
    )
    resume_final_rollup = (
        prior_rollup_metadata.get("rollup_signature") == rollup_signature
    )
if resume_final_rollup:
    participant_visit_embeddings = pd.read_parquet(participant_visit_embedding_path)
    print("Resumed final participant-visit embedding rollup")
else:
    accumulators = {}
    for index, path in enumerate(rollup_paths, 1):
        batch = pd.read_parquet(path)
        for record in batch.itertuples(index=False):
            key = (str(record.participant_id), str(record.visit))
            vector = np.asarray(record.embedding_sum, dtype=np.float64).reshape(-1)
            count = int(record.n_embedded_images)
            if key not in accumulators:
                accumulators[key] = [vector.copy(), count]
            else:
                accumulators[key][0] += vector
                accumulators[key][1] += count
        if index == 1 or index % 25 == 0 or index == len(rollup_paths):
            print(f"[combine {index}/{len(rollup_paths)}] {len(accumulators):,} participant-visits")
    participant_visit_embeddings = pd.DataFrame(
        [
            {
                "participant_id": participant_id,
                "visit": visit,
                "embedding": (vector_sum / count).astype(np.float32),
                "n_embedded_images": count,
            }
            for (participant_id, visit), (vector_sum, count) in accumulators.items()
        ]
    )
    local = Path("/local_disk0/tmp") / f"fairness-final-rollup-{uuid.uuid4().hex}.parquet"
    write_frame(participant_visit_embeddings, local)
    publish_local_file(local, participant_visit_embedding_path)
    local.unlink(missing_ok=True)
    del accumulators
    write_json(
        {
            "rollup_signature": rollup_signature,
            "n_embedding_parquets": len(embedding_paths),
            "n_successfully_embedded_images": len(embedded_paths & quality_pass_paths),
        },
        participant_visit_embedding_metadata_path,
    )

if participant_visit_embeddings.duplicated(["participant_id", "visit"]).any():
    raise ValueError("Participant-visit embedding rollup is not unique")
dimensions = participant_visit_embeddings["embedding"].map(
    lambda value: np.asarray(value).reshape(-1).size
)
if set(dimensions) != {expected_embedding_dim}:
    raise ValueError(f"Unexpected final embedding dimensions: {sorted(set(dimensions))}")
print(
    f"Final rollup: {len(participant_visit_embeddings):,} participant-visits, "
    f"{participant_visit_embeddings['participant_id'].nunique():,} participants, "
    f"{participant_visit_embeddings['n_embedded_images'].sum():,} images represented"
)
if int(participant_visit_embeddings["n_embedded_images"].sum()) != len(
    embedded_paths & quality_pass_paths
):
    raise RuntimeError(
        "Participant-visit rollup image count does not match the complete embedding ledger"
    )


## 3. Load visit-matched age, demographics, and comorbidities


In [ ]:
required_sap_columns = ["participant_id", "visit", "age_at_fundus_years"]
candidate_sap_columns = [
    "sex_at_birth",
    "education_level_sap_harmonized",
    "household_income_band",
    "diabetes",
    "hypertension",
    "heart_disease",
    "stroke",
    "arthritis_any",
    "osteoporosis",
    "asthma_or_copd",
    "cancer",
    "low_back_pain",
    "depression_cesd10",
    "smoking_status",
    "multimorbidity_selected_count",
]
sap_spark = spark.read.format("delta").load(str(sap_questionnaire_path))
missing_sap = set(required_sap_columns) - set(sap_spark.columns)
if missing_sap:
    raise ValueError(f"SAP table missing required fields: {sorted(missing_sap)}")
available_sap_columns = required_sap_columns + [
    column for column in candidate_sap_columns if column in sap_spark.columns
]
sap = sap_spark.select(*available_sap_columns).toPandas()
sap.attrs = {}
sap["participant_id"] = normalize_identifier(sap["participant_id"])
sap["visit"] = normalize_visit(sap["visit"])
sap["age_at_fundus_years"] = pd.to_numeric(sap["age_at_fundus_years"], errors="coerce")

def stable_value(series):
    values = series.dropna()
    if values.empty:
        return None
    mode = values.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else values.iloc[0]

sap_aggregations = {"age_at_fundus_years": "median"}
for column in available_sap_columns:
    if column not in {"participant_id", "visit", "age_at_fundus_years"}:
        sap_aggregations[column] = stable_value
sap = sap.groupby(["participant_id", "visit"], as_index=False).agg(sap_aggregations)
if sap.duplicated(["participant_id", "visit"]).any():
    raise ValueError("Collapsed SAP table remains non-unique")
print("Available matching/demographic fields:", available_sap_columns[3:])


### Released self-reported cultural/racial background

These baseline variables are multiple-response indicators. The code reads only
participants appearing in the fundus quality manifest, leaves unknown/refused
values missing, and never coerces a multiple selection into one category.


In [ ]:
RACIAL_BACKGROUND_FIELDS = {
    "SDC_CULT_WH_COM": "White",
    "SDC_CULT_ZH_COM": "Chinese",
    "SDC_CULT_SA_COM": "South Asian",
    "SDC_CULT_BL_COM": "Black",
    "SDC_CULT_FP_COM": "Filipino",
    "SDC_CULT_LA_COM": "Latin American",
    "SDC_CULT_SE_COM": "Southeast Asian",
    "SDC_CULT_AR_COM": "Arab",
    "SDC_CULT_WA_COM": "West Asian",
    "SDC_CULT_JA_COM": "Japanese",
    "SDC_CULT_KO_COM": "Korean",
    "SDC_CULT_OT_COM": "Other",
}

with zipfile.ZipFile(baseline_archive_path) as archive:
    matches = [name for name in archive.namelist() if name.endswith(baseline_member_suffix)]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one baseline CSV ending in {baseline_member_suffix!r}; found {len(matches)}"
        )
    baseline_member = matches[0]
    with archive.open(baseline_member) as stream:
        header = pd.read_csv(stream, nrows=0).columns.tolist()

participant_candidates = [
    "entity_id", "participant_id", "ID", "id", "Entity_ID", "ENTITY_ID"
]
baseline_id_column = next((column for column in participant_candidates if column in header), None)
if baseline_id_column is None:
    raise ValueError("Could not identify the participant ID column in baseline CSV")
missing_race_fields = set(RACIAL_BACKGROUND_FIELDS) - set(header)
if missing_race_fields:
    raise ValueError(
        "Baseline CSV is missing documented cultural/racial fields: "
        f"{sorted(missing_race_fields)}"
    )

fundus_participant_ids = set(quality["participant_id"].dropna().astype(str))
race_chunks = []
with zipfile.ZipFile(baseline_archive_path) as archive:
    with archive.open(baseline_member) as stream:
        for chunk_index, chunk in enumerate(
            pd.read_csv(
                stream,
                usecols=[baseline_id_column, *RACIAL_BACKGROUND_FIELDS],
                dtype="string",
                chunksize=100_000,
                low_memory=False,
            ),
            1,
        ):
            chunk[baseline_id_column] = normalize_identifier(chunk[baseline_id_column])
            selected = chunk[chunk[baseline_id_column].isin(fundus_participant_ids)].copy()
            if not selected.empty:
                race_chunks.append(selected)
            print(f"[baseline race chunk {chunk_index}] retained {len(selected):,}")

if not race_chunks:
    raise ValueError("No fundus participants linked to the baseline racial-background fields")
race = pd.concat(race_chunks, ignore_index=True).rename(
    columns={baseline_id_column: "participant_id"}
)
race = race.drop_duplicates("participant_id", keep="last")
race = classify_racial_background(race, RACIAL_BACKGROUND_FIELDS)
raw_any_selection = race[list(RACIAL_BACKGROUND_FIELDS)].apply(
    lambda column: column.astype("string").str.strip().isin(["1", "1.0"])
).any(axis=1)
decoded_any_selection = race["racial_background_analysis_eligible"].fillna(False)
if raw_any_selection.any() and not decoded_any_selection.any():
    raise RuntimeError(
        "Racial-background indicators contain selected values but none decoded. "
        "Inspect retfound_fairness.binary_indicator before continuing."
    )
if (raw_any_selection & ~decoded_any_selection).any():
    raise RuntimeError(
        f"{int((raw_any_selection & ~decoded_any_selection).sum()):,} participants "
        "have a selected racial-background indicator but no decoded category."
    )
print(
    f"Decoded racial/cultural background for {int(decoded_any_selection.sum()):,}/"
    f"{len(race):,} linked baseline participants"
)
write_frame(race, private_root / "racial_background_private.parquet")

race_counts = (
    race.groupby(["racial_background", "racial_background_status"], dropna=False)
    .agg(n_participants=("participant_id", "nunique"))
    .reset_index()
    .sort_values("n_participants", ascending=False)
)
race_counts_reportable = race_counts[
    race_counts["n_participants"] >= minimum_reporting_group_n
].copy()
suppressed_count = int(
    race_counts.loc[
        race_counts["n_participants"] < minimum_reporting_group_n,
        "n_participants",
    ].sum()
)
if suppressed_count:
    race_counts_reportable = pd.concat(
        [
            race_counts_reportable,
            pd.DataFrame(
                [{
                    "racial_background": "Suppressed small groups combined",
                    "racial_background_status": "suppressed",
                    "n_participants": suppressed_count,
                }]
            ),
        ],
        ignore_index=True,
    )
display(race_counts_reportable)


## 4. Quality-selection fairness

This is evaluated **before** restricting the age model to passing images.
Otherwise a demographic disparity introduced by technical-quality filtering
would be invisible. The participant endpoint is whether at least one image passed.


In [ ]:
quality_race = quality.merge(
    race[["participant_id", "racial_background"]],
    on="participant_id",
    how="left",
    validate="many_to_one",
)
participant_quality = (
    quality_race.groupby("participant_id", as_index=False)
    .agg(
        racial_background=("racial_background", stable_value),
        n_images=("image_path", "size"),
        n_passing_images=("quality_pass_bool", "sum"),
        any_image_passed=("quality_pass_bool", "max"),
    )
)
quality_rows = []
for group, subset in participant_quality.dropna(subset=["racial_background"]).groupby("racial_background"):
    passed = int(subset["any_image_passed"].sum())
    total = int(len(subset))
    low, high = proportion_confint(passed, total, alpha=0.05, method="wilson")
    quality_rows.append(
        {
            "racial_background": group,
            "n_participants": total,
            "participants_with_passing_image": passed,
            "participant_pass_rate": passed / total,
            "pass_rate_ci_low": low,
            "pass_rate_ci_high": high,
            "mean_passing_images": float(subset["n_passing_images"].mean()),
        }
    )
quality_summary = pd.DataFrame(quality_rows).sort_values("n_participants", ascending=False)
write_frame(quality_summary, private_root / "quality_selection_by_race_private.parquet")
quality_summary_reportable = quality_summary[
    quality_summary["n_participants"] >= minimum_reporting_group_n
].copy()
write_frame(
    quality_summary_reportable,
    statistics_root / "quality_selection_by_race.parquet",
)
display(quality_summary_reportable)


## 5. Train/resume one CLSA-wide retinal-age head

The training population includes every participant-visit with at least one
quality-passing RETFound vector and visit-matched chronological age. Every
successfully embedded eye/image contributes to its participant-visit mean; both
BL and F1 are retained. No race or demographic field enters the Ridge model,
and OOF folds are grouped by participant so BL/F1 and both eyes never leak
across training and testing.
The final frozen model is saved as `CLSA_full_cohort_age_head.joblib`; fairness
conclusions use OOF predictions, not its in-sample fitted values.


In [ ]:
participant_visit = participant_visit_embeddings.merge(
    sap,
    on=["participant_id", "visit"],
    how="left",
    validate="one_to_one",
)
participant_visit = participant_visit.rename(columns={"age_at_fundus_years": "age"})
participant_visit["age"] = pd.to_numeric(participant_visit["age"], errors="coerce")
training = participant_visit.dropna(subset=["age", "embedding"]).reset_index(drop=True)
if training.duplicated(["participant_id", "visit"]).any():
    raise ValueError("Age training frame is not unique by participant and visit")

signature_text = "\n".join(
    training[["participant_id", "visit", "age", "n_embedded_images"]]
    .astype(str)
    .agg("|".join, axis=1)
    .sort_values()
)
training_signature = hashlib.sha256(signature_text.encode("utf-8")).hexdigest()
model_path = model_root / "CLSA_full_cohort_age_head.joblib"
oof_path = model_root / "CLSA_full_cohort_age_predictions_oof.parquet"
metadata_path = model_root / "CLSA_full_cohort_age_model_metadata.json"

can_resume = False
if resume_completed_outputs and model_path.exists() and oof_path.exists() and metadata_path.exists():
    saved_metadata = json.loads(metadata_path.read_text())
    can_resume = saved_metadata.get("training_signature") == training_signature

if can_resume:
    age_bundle = joblib.load(model_path)
    age_oof = pd.read_parquet(oof_path)
    print("Resumed matching CLSA-wide age model and exact OOF predictions")
else:
    local_training_root = Path("/local_disk0/tmp") / f"fairness-age-model-{uuid.uuid4().hex}"
    local_training_root.mkdir(parents=True, exist_ok=True)
    age_oof, age_bundle = train_age_head(
        training[["participant_id", "visit", "age", "embedding", "n_embedded_images"]],
        local_training_root,
        AgeModelConfig(
            alpha=ridge_alpha,
            max_splits=cv_folds,
            calibration="intercept",
            random_state=random_state,
        ),
        write_metadata=False,
    )
    age_bundle.update(
        {
            "model_name": "CLSA_full_cohort_age_head",
            "training_signature": training_signature,
            "training_population": "all quality-passing CLSA participant-visits with age",
        }
    )
    local_model = local_training_root / "CLSA_full_cohort_age_head.joblib"
    joblib.dump(age_bundle, local_model)
    local_oof = local_training_root / "CLSA_full_cohort_age_predictions_oof.parquet"
    write_frame(age_oof, local_oof)
    publish_local_file(local_model, model_path)
    publish_local_file(local_oof, oof_path)
    for filename in ("retfound_age_fold_diagnostics.csv", "retfound_age_metrics.csv"):
        source = local_training_root / filename
        if source.exists():
            publish_local_file(source, model_root / filename)
    write_json(
        {
            "training_signature": training_signature,
            "n_training_participants": int(training["participant_id"].nunique()),
            "n_training_participant_visits": int(len(training)),
            "n_training_images_represented": int(training["n_embedded_images"].sum()),
            "embedding_dim": expected_embedding_dim,
            "ridge_alpha": ridge_alpha,
            "cv_folds": cv_folds,
            "model_path": str(model_path),
            "oof_path": str(oof_path),
            "race_used_as_model_input": False,
        },
        metadata_path,
    )
    shutil.rmtree(local_training_root, ignore_errors=True)

if age_oof.duplicated(["participant_id", "visit"]).any():
    raise ValueError("OOF predictions are not unique by participant and visit")
print(
    f"Age head: {len(age_oof):,} OOF participant-visits from "
    f"{age_oof['participant_id'].nunique():,} participants; "
    f"MAE={age_oof['absolute_error_oof'].mean():.2f} years; "
    f"mean gap={age_oof['retinal_age_gap_oof'].mean():.2f} years"
)


## 6. Participant-level demographic performance


In [ ]:
analysis = pool_age_predictions_to_participants(
    age_oof,
    participant_column="participant_id",
    visit_column="visit",
    age_column="age",
    prediction_column="retinal_age_prediction_oof",
    visit_priority=("BL", "F1"),
)
analysis["participant_id"] = normalize_identifier(analysis["participant_id"])
analysis = analysis.merge(
    sap.drop(columns=["age_at_fundus_years"], errors="ignore"),
    on=["participant_id", "visit"],
    how="left",
    validate="one_to_one",
)
analysis = analysis.merge(
    race[[
        "participant_id",
        "racial_background",
        "racial_background_detail",
        "racial_background_status",
    ]],
    on="participant_id",
    how="left",
    validate="one_to_one",
)
analysis["age"] = pd.to_numeric(analysis["age"], errors="coerce")
analysis["retinal_age_gap_oof"] = (
    analysis["retinal_age_prediction_oof"] - analysis["age"]
)
analysis["absolute_error_oof"] = analysis["retinal_age_gap_oof"].abs()
write_frame(analysis, private_root / "fairness_participant_analysis_private.parquet")

demographic_columns = [
    column
    for column in (
        "racial_background",
        "sex_at_birth",
        "education_level_sap_harmonized",
        "household_income_band",
        "visit",
    )
    if column in analysis.columns
]
metric_frames = []
for index, demographic in enumerate(demographic_columns):
    metrics = fairness_metric_table(
        analysis,
        demographic,
        bootstrap_repetitions=bootstrap_repetitions,
        random_state=random_state + index,
    )
    metrics = metrics.rename(columns={demographic: "demographic_level"})
    metrics.insert(0, "demographic", demographic)
    metric_frames.append(metrics)
demographic_metrics = pd.concat(metric_frames, ignore_index=True)
demographic_metrics["reportable"] = (
    demographic_metrics["n_participants"] >= minimum_reporting_group_n
)
write_frame(
    demographic_metrics,
    private_root / "demographic_age_performance_all_cells_private.parquet",
)
demographic_metrics_reportable = demographic_metrics[
    demographic_metrics["reportable"]
].copy()
write_frame(
    demographic_metrics_reportable,
    statistics_root / "demographic_age_performance.parquet",
)
display(demographic_metrics_reportable)


## 7. Race-specific comorbidity matching

Primary matching uses a ±1-year age caliper, exact sex, and nearest released
comorbidity profile. References are not reused within a race-specific comparison.
The ±2-year run is a sensitivity analysis. White participants may appear in
different race-specific comparisons because each comparison estimates a distinct
group-versus-White contrast. Socioeconomic measures are reported as fairness
strata but are not primary matching covariates, avoiding overmatching on possible
social mediators.


In [ ]:
candidate_distance_columns = [
    "diabetes",
    "hypertension",
    "heart_disease",
    "stroke",
    "arthritis_any",
    "osteoporosis",
    "asthma_or_copd",
    "cancer",
    "low_back_pain",
    "depression_cesd10",
    "smoking_status",
    "multimorbidity_selected_count",
]
distance_columns = [column for column in candidate_distance_columns if column in analysis.columns]
numeric_distance_columns = [
    column for column in ("multimorbidity_selected_count",) if column in distance_columns
]
exact_columns = [column for column in ("sex_at_birth",) if column in analysis.columns]

race_sizes = analysis["racial_background"].value_counts(dropna=True)
target_groups = [
    group
    for group, count in race_sizes.items()
    if group not in {"White", "Multiple groups"}
    and count >= minimum_inference_group_n
]
if "White" not in race_sizes:
    raise ValueError("No analyzable White reference group was found")
if not target_groups:
    raise ValueError(
        "No non-White released group meets the prespecified inference threshold; "
        f"counts were {race_sizes.to_dict()}"
    )

all_pairs = []
all_audits = []
all_memberships = []
all_contrasts = []
all_balance = []
for caliper_label, caliper in (
    ("primary_1y", primary_age_caliper_years),
    ("sensitivity_2y", sensitivity_age_caliper_years),
):
    for group_index, target_group in enumerate(target_groups):
        pairs, audit, membership = match_group_to_reference(
            analysis,
            str(target_group),
            group_column="racial_background",
            reference_group="White",
            age_caliper_years=caliper,
            ratio=match_ratio,
            exact_columns=exact_columns,
            distance_columns=distance_columns,
            numeric_distance_columns=numeric_distance_columns,
        )
        for frame in (pairs, audit, membership):
            frame["caliper_analysis"] = caliper_label
            frame["age_caliper_years"] = caliper
        all_pairs.append(pairs)
        all_audits.append(audit)
        all_memberships.append(membership)

        contrasts = matched_outcome_contrasts(
            analysis,
            membership,
            bootstrap_repetitions=max(bootstrap_repetitions, 2000),
            random_state=random_state + group_index,
        )
        contrasts["caliper_analysis"] = caliper_label
        contrasts["age_caliper_years"] = caliper
        all_contrasts.append(contrasts)

        if not membership.empty:
            matched = membership.merge(
                analysis,
                on="participant_id",
                how="left",
                validate="many_to_one",
            )
            matched_target = matched[matched["match_role"] == "target"]
            matched_reference = matched[matched["match_role"] == "reference"]
            before = standardized_mean_differences(
                analysis[analysis["racial_background"] == target_group],
                analysis[analysis["racial_background"] == "White"],
                ["age", *exact_columns, *distance_columns],
            )
            before["phase"] = "Before matching"
            after = standardized_mean_differences(
                matched_target,
                matched_reference,
                ["age", *exact_columns, *distance_columns],
            )
            after["phase"] = "After matching"
            balance = pd.concat([before, after], ignore_index=True)
            balance["target_group"] = target_group
            balance["caliper_analysis"] = caliper_label
            all_balance.append(balance)

match_pairs = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame()
match_audit = pd.concat(all_audits, ignore_index=True) if all_audits else pd.DataFrame()
match_membership = pd.concat(all_memberships, ignore_index=True) if all_memberships else pd.DataFrame()
matched_contrasts = pd.concat(all_contrasts, ignore_index=True) if all_contrasts else pd.DataFrame()
balance = pd.concat(all_balance, ignore_index=True) if all_balance else pd.DataFrame()

if not matched_contrasts.empty:
    matched_contrasts["p_value_fdr"] = matched_contrasts.groupby(
        ["caliper_analysis", "outcome"]
    )["p_value"].transform(lambda values: benjamini_hochberg(values.to_numpy()))

# Identifier-bearing artifacts remain private.
write_frame(match_pairs, private_root / "race_match_pairs_private.parquet")
write_frame(match_audit, private_root / "race_match_audit_private.parquet")
write_frame(match_membership, private_root / "race_match_membership_private.parquet")
write_frame(matched_contrasts, statistics_root / "race_matched_outcome_contrasts.parquet")
write_frame(balance, statistics_root / "race_matching_balance.parquet")

match_summary = (
    match_audit.groupby(["caliper_analysis", "target_group"], as_index=False)
    .agg(
        target_participants=("target_participant_id", "size"),
        matched_targets=("matched", "sum"),
        total_matched_references=("matched_reference_count", "sum"),
    )
)
match_summary["target_match_rate"] = (
    match_summary["matched_targets"] / match_summary["target_participants"]
)
write_frame(match_summary, statistics_root / "race_matching_summary.parquet")
display(match_summary)
display(matched_contrasts)


## 8. Publication figures


In [ ]:
# Figure 1: cohort and quality selection.
reportable_quality = quality_summary[
    quality_summary["n_participants"] >= minimum_reporting_group_n
].sort_values("participant_pass_rate")
figure, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(
    data=reportable_quality,
    y="racial_background",
    x="n_participants",
    color="#4C78A8",
    ax=axes[0],
)
axes[0].set(title="Fundus participants with released group", xlabel="Participants", ylabel="")
axes[1].errorbar(
    reportable_quality["participant_pass_rate"],
    np.arange(len(reportable_quality)),
    xerr=np.vstack([
        reportable_quality["participant_pass_rate"] - reportable_quality["pass_rate_ci_low"],
        reportable_quality["pass_rate_ci_high"] - reportable_quality["participant_pass_rate"],
    ]),
    fmt="o",
    color="#E45756",
    capsize=3,
)
axes[1].set_yticks(np.arange(len(reportable_quality)), reportable_quality["racial_background"])
axes[1].set(xlabel="At least one quality-passing image (95% CI)", ylabel="", xlim=(0, 1.02))
axes[1].set_title("Participant-level quality selection")
figure.suptitle("CLSA RETFound fairness cohort and preprocessing audit")
figure.tight_layout()
figure.savefig(figure_root / "figure_1_cohort_quality_selection.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 2: global OOF calibration and race-specific error estimates.
race_metrics = demographic_metrics[
    (demographic_metrics["demographic"] == "racial_background")
    & (demographic_metrics["n_participants"] >= minimum_reporting_group_n)
].copy()
race_metrics = race_metrics[race_metrics["demographic_level"] != "Overall"].sort_values("mae")

figure, axes = plt.subplots(1, 3, figsize=(22, 6))
hexbin = axes[0].hexbin(
    analysis["age"],
    analysis["retinal_age_prediction_oof"],
    gridsize=35,
    mincnt=1,
    cmap="viridis",
)
limits = [analysis["age"].min() - 2, analysis["age"].max() + 2]
axes[0].plot(limits, limits, "--", color="white", linewidth=2)
axes[0].set(xlabel="Chronological age (years)", ylabel="OOF retinal age (years)", title="Global CLSA age head")
figure.colorbar(hexbin, ax=axes[0], label="Participants per hex")

for axis, metric, low, high, title in (
    (axes[1], "mean_gap", "mean_gap_ci_low", "mean_gap_ci_high", "Mean retinal-age gap"),
    (axes[2], "mae", "mae_ci_low", "mae_ci_high", "Mean absolute error"),
):
    plot = race_metrics.sort_values(metric).reset_index(drop=True)
    axis.errorbar(
        plot[metric],
        np.arange(len(plot)),
        xerr=np.vstack([plot[metric] - plot[low], plot[high] - plot[metric]]),
        fmt="o",
        capsize=3,
        color="#4C78A8",
    )
    axis.set_yticks(np.arange(len(plot)), plot["demographic_level"])
    axis.set(title=title, xlabel="Years (95% bootstrap CI)", ylabel="")
    if metric == "mean_gap":
        axis.axvline(0, color="black", linestyle="--", linewidth=1)
figure.tight_layout()
figure.savefig(figure_root / "figure_2_oof_age_performance_by_race.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 3: primary matched contrasts and matching balance.
primary_contrasts = matched_contrasts[
    matched_contrasts["caliper_analysis"] == "primary_1y"
].copy()
primary_balance = (
    balance[balance["caliper_analysis"] == "primary_1y"].copy()
    if "caliper_analysis" in balance.columns
    else pd.DataFrame()
)
figure, axes = plt.subplots(1, 2, figsize=(18, max(6, len(primary_contrasts) * 0.45)))
if not primary_contrasts.empty:
    contrast_plot = primary_contrasts.sort_values(["outcome", "target_group"]).reset_index(drop=True)
    labels = contrast_plot["target_group"] + " — " + contrast_plot["outcome"].replace(
        {"retinal_age_gap_oof": "age gap", "absolute_error_oof": "absolute error"}
    )
    axes[0].errorbar(
        contrast_plot["target_minus_reference"],
        np.arange(len(contrast_plot)),
        xerr=np.vstack([
            contrast_plot["target_minus_reference"] - contrast_plot["ci_low"],
            contrast_plot["ci_high"] - contrast_plot["target_minus_reference"],
        ]),
        fmt="o",
        capsize=3,
        color="#E45756",
    )
    axes[0].set_yticks(np.arange(len(contrast_plot)), labels)
    axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
    axes[0].set(
        title="Matched target minus White reference",
        xlabel="Difference in years (95% bootstrap CI)",
        ylabel="",
    )
if not primary_balance.empty:
    balance_plot = (
        primary_balance.groupby(["phase", "covariate"], as_index=False)["abs_smd"].max()
    )
    sns.scatterplot(
        data=balance_plot,
        x="abs_smd",
        y="covariate",
        hue="phase",
        style="phase",
        s=90,
        ax=axes[1],
    )
    axes[1].axvline(0.1, color="black", linestyle="--", linewidth=1)
    axes[1].set(title="Worst balance across race comparisons", xlabel="Absolute standardized mean difference", ylabel="")
figure.tight_layout()
figure.savefig(figure_root / "figure_3_matched_race_contrasts_balance.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 4: secondary demographic fairness panels.
secondary = demographic_metrics[
    demographic_metrics["demographic"].isin([
        "sex_at_birth", "education_level_sap_harmonized", "household_income_band"
    ])
    & (demographic_metrics["demographic_level"] != "Overall")
    & (demographic_metrics["n_participants"] >= minimum_reporting_group_n)
].copy()
if not secondary.empty:
    demographics = secondary["demographic"].unique().tolist()
    figure, axes = plt.subplots(1, len(demographics), figsize=(8 * len(demographics), 6), squeeze=False)
    for axis, demographic in zip(axes.ravel(), demographics):
        plot = secondary[secondary["demographic"] == demographic].sort_values("mean_gap").reset_index(drop=True)
        axis.errorbar(
            plot["mean_gap"],
            np.arange(len(plot)),
            xerr=np.vstack([
                plot["mean_gap"] - plot["mean_gap_ci_low"],
                plot["mean_gap_ci_high"] - plot["mean_gap"],
            ]),
            fmt="o",
            capsize=3,
        )
        axis.set_yticks(np.arange(len(plot)), plot["demographic_level"])
        axis.axvline(0, color="black", linestyle="--", linewidth=1)
        axis.set(title=demographic.replace("_", " ").title(), xlabel="Mean OOF retinal-age gap (years)", ylabel="")
    figure.tight_layout()
    figure.savefig(figure_root / "figure_4_secondary_demographic_age_gap.png", dpi=220, bbox_inches="tight")
    plt.show()


## 9. Cohort flow, diagnostics, and generated run README


In [ ]:
flow = pd.DataFrame(
    [
        {"stage": "Participants in complete fundus manifest", "n_participants": full_manifest["participant_id"].nunique()},
        {"stage": "At least one quality-passing image", "n_participants": participant_quality["any_image_passed"].sum()},
        {"stage": "Completed RETFound participant embedding", "n_participants": participant_visit_embeddings["participant_id"].nunique()},
        {"stage": "Grouped-CV age model", "n_participants": training["participant_id"].nunique()},
        {"stage": "Released single/multiple racial background", "n_participants": analysis["racial_background"].notna().sum()},
    ]
)
write_frame(flow, statistics_root / "cohort_flow.parquet")

max_postmatch_smd = (
    balance.loc[balance["phase"] == "After matching", "abs_smd"].max()
    if not balance.empty
    else np.nan
)
readme = f'''# CLSA RETFound algorithm fairness run

## Scope

This run evaluates technical-quality selection and a single CLSA-wide RETFound
retinal-age head. Race/demographic variables were not model inputs. All reported
age performance is participant-level and out-of-fold.

## Cohort

- Raw images in complete manifest: {len(full_manifest_paths):,}
- Participants in complete fundus manifest: {full_manifest['participant_id'].nunique():,}
- Images entering technical-quality processing: {len(quality_paths):,}
- Images passing technical quality: {len(quality_pass_paths):,}
- Images successfully represented by RETFound: {len(embedded_paths & quality_pass_paths):,}
- Participants with at least one quality-passing image: {int(participant_quality['any_image_passed'].sum()):,}
- Participant-visits in grouped-CV retinal-age model: {len(age_oof):,}
- Unique participants in grouped-CV retinal-age model: {age_oof['participant_id'].nunique():,}
- Successfully embedded images contributing to the model: {int(training['n_embedded_images'].sum()):,}
- Participants with an analyzable released racial-background category: {int(analysis['racial_background'].notna().sum()):,}
- Participant-level OOF MAE: {analysis['absolute_error_oof'].mean():.2f} years
- Participant-level mean OOF retinal-age gap: {analysis['retinal_age_gap_oof'].mean():.2f} years

## Prespecified comparisons

- Reference category: White
- Primary age caliper: ±{primary_age_caliper_years:g} year
- Sensitivity caliper: ±{sensitivity_age_caliper_years:g} years
- Requested reference ratio: 1:{match_ratio}
- Exact match fields: {', '.join(exact_columns) if exact_columns else 'none'}
- Distance fields: {', '.join(distance_columns)}
- Maximum post-match absolute SMD: {max_postmatch_smd:.3f}
- Inference threshold: at least {minimum_inference_group_n} participants per released group
- Bootstrap repetitions: {bootstrap_repetitions}

## Interpretation guardrails

The released categories are self-reported cultural/racial backgrounds, not
genetic ancestry. `Black` should not be relabeled `African`. A disparity is
evidence about the observed CLSA pipeline—not proof that the pretrained encoder
alone is inherently biased. Quality selection, camera/site, enrollment,
comorbidity measurement, chronological-age measurement, and residual imbalance
remain alternative explanations. Inspect match rates and post-match SMDs before
interpreting outcome contrasts. Groups below the privacy/reporting threshold are
not shown in figures, and groups below the inference threshold are descriptive only.

Identifier-bearing match and race tables are stored only under `01_private`.
'''
(output_root / "RUN_README.md").write_text(readme, encoding="utf-8")
write_json(
    {
        "completed": True,
        "training_signature": training_signature,
        "n_full_manifest_images": len(full_manifest_paths),
        "n_quality_attempted_images": len(quality_paths),
        "n_embedded_images": len(embedded_paths & quality_pass_paths),
        "n_age_model_participant_visits": len(age_oof),
        "n_age_model_participants": int(age_oof["participant_id"].nunique()),
        "target_groups": target_groups,
        "quality_path": str(quality_path),
        "embedding_source_mode": embedding_source_mode,
        "output_root": str(output_root),
    },
    output_root / "run_manifest.json",
)
display(flow)
print(readme)
print("Completed fairness workflow:", output_root)


## Interpretation checklist

- Use the **quality-selection** figure before interpreting the age model.
- Require adequate group size, overlap, match rate, and post-match balance
  (ideally absolute SMD <0.10).
- Treat the 1-year match as primary and the 2-year result as sensitivity.
- Distinguish systematic signed error (mean age gap) from accuracy (MAE/RMSE)
  and calibration slope.
- Do not use in-sample predictions or count two eyes as independent subjects.
- Avoid causal language. A model can have equal aggregate MAE yet different
  calibration or signed errors, and vice versa.


## 10. All-available race-specific age heads with matched White comparators

This is the power-enhanced subgroup analysis. It does **not** force every group
down to the smallest sample. Every participant in each specific released group
that met the prespecified inference threshold and received a primary ±1-year
match is retained. `Other` and `Multiple groups` are excluded because they are
not single specific categories.

For each group, two separate Ridge heads are fit: one from all target-group
participants and one from that group's 1:2 age/sex/comorbidity-matched White
cohort. Five-fold participant cross-fitting gives every source-population
participant an out-of-fold prediction. Final heads fit on all corresponding
participants are saved for notebook 02 spatial explainability. One completed
RETFound image is used per participant; participants, not eyes, are the unit of
inference.


In [ ]:
# All-available race-specific RETFound age heads against matched White cohorts
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import norm

all_available_model_root = model_root / "race_matched_all_available_age_heads"
all_available_statistics_root = statistics_root / "race_matched_all_available_age_heads"
all_available_figure_root = figure_root / "race_matched_all_available_age_heads"
for directory in (all_available_model_root, all_available_statistics_root, all_available_figure_root):
    directory.mkdir(parents=True, exist_ok=True)

all_available_seed = random_state + 1201
all_available_folds = 5
all_available_bootstraps = 2000
minimum_all_available_n = minimum_inference_group_n
excluded_nonspecific_groups = {"White", "Multiple groups", "Other"}


def safe_group_name(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()


def fit_intercept_calibrated_ridge(frame):
    matrix = np.stack(frame["embedding"].map(lambda value: np.asarray(value, dtype=np.float32)))
    age_values = frame["age"].to_numpy(float)
    estimator = Ridge(alpha=ridge_alpha).fit(matrix, age_values)
    offset = float(np.mean(age_values - estimator.predict(matrix)))
    return estimator, offset


def crossfit_population(frame, seed):
    frame = frame.sort_values("participant_id", kind="stable").reset_index(drop=True).copy()
    if frame["participant_id"].duplicated().any():
        raise ValueError("Cross-fitting requires one image per participant")
    n_splits = min(all_available_folds, len(frame) // 6)
    if n_splits < 3:
        raise ValueError(f"At least 18 participants are required; found {len(frame)}")
    folds = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    prediction = np.full(len(frame), np.nan)
    fold_number = np.full(len(frame), -1, dtype=int)
    for fold, (train_index, test_index) in enumerate(folds.split(frame)):
        estimator, offset = fit_intercept_calibrated_ridge(frame.iloc[train_index])
        matrix = np.stack(frame.iloc[test_index]["embedding"].map(lambda value: np.asarray(value, dtype=np.float32)))
        prediction[test_index] = estimator.predict(matrix) + offset
        fold_number[test_index] = fold
    if not np.isfinite(prediction).all() or (fold_number < 0).any():
        raise RuntimeError("Cross-fitting did not assign every participant")
    final_estimator, final_offset = fit_intercept_calibrated_ridge(frame)
    return prediction, fold_number, final_estimator, final_offset, n_splits


# Use only fully completed primary matches and specific released groups.
primary_membership = match_membership[
    match_membership["caliper_analysis"].astype(str).eq("primary_1y")
].copy()
eligible_specific_groups = sorted(
    group for group in primary_membership["target_group"].dropna().astype(str).unique()
    if group not in excluded_nonspecific_groups
    and int((analysis["racial_background"].astype(str) == group).sum()) >= minimum_all_available_n
)
if not eligible_specific_groups:
    raise ValueError("No specific released racial-background groups meet the inference threshold")
primary_membership = primary_membership[
    primary_membership["target_group"].astype(str).isin(eligible_specific_groups)
].copy()
if primary_membership.empty:
    raise ValueError("Primary matched membership is empty")

# Build one stable completed-image lookup for the union of matched participants.
participant_ids_needed = set(primary_membership["participant_id"].astype(str))
participant_endpoint = analysis[["participant_id", "visit", "age", "racial_background"]].copy()
participant_endpoint["participant_id"] = normalize_identifier(participant_endpoint["participant_id"])
participant_endpoint["visit"] = normalize_visit(participant_endpoint["visit"])
participant_endpoint = participant_endpoint[participant_endpoint["participant_id"].isin(participant_ids_needed)]
if participant_endpoint["participant_id"].duplicated().any():
    raise ValueError("Participant endpoint table is not unique")

ledger_frames = []
for path_index, embedding_path in enumerate(embedding_paths, 1):
    ledger = pd.read_parquet(embedding_path, columns=["image_path", "participant_id", "visit"])
    ledger["participant_id"] = normalize_identifier(ledger["participant_id"])
    ledger["visit"] = normalize_visit(ledger["visit"])
    ledger = ledger[ledger["participant_id"].isin(participant_ids_needed)]
    if not ledger.empty:
        ledger_frames.append(ledger)
    if path_index == 1 or path_index % 25 == 0 or path_index == len(embedding_paths):
        print(f"[all-available ledger {path_index}/{len(embedding_paths)}]", flush=True)
if not ledger_frames:
    raise ValueError("No completed embeddings linked to matched participants")
image_candidates = pd.concat(ledger_frames, ignore_index=True).drop_duplicates("image_path", keep="last")
image_candidates = image_candidates.merge(
    participant_endpoint.rename(columns={"visit": "selected_visit"}),
    on="participant_id", how="inner", validate="many_to_one",
)
image_candidates = image_candidates[image_candidates["visit"] == image_candidates["selected_visit"]].copy()
image_candidates["_stable_random"] = image_candidates["image_path"].map(
    lambda value: int(hashlib.sha256(f"{all_available_seed}|{value}".encode()).hexdigest()[:16], 16)
)
participant_image_lookup = (
    image_candidates.sort_values(["participant_id", "_stable_random", "image_path"], kind="stable")
    .drop_duplicates("participant_id", keep="first")
    .drop(columns="_stable_random")
)
missing_images = participant_ids_needed - set(participant_image_lookup["participant_id"])
if missing_images:
    raise RuntimeError(f"{len(missing_images)} matched participants lack a completed vector at the endpoint visit")

selected_paths = set(participant_image_lookup["image_path"].astype(str))
vector_frames = []
for path_index, embedding_path in enumerate(embedding_paths, 1):
    frame = pd.read_parquet(embedding_path)
    frame = frame[frame["image_path"].astype(str).isin(selected_paths)]
    if not frame.empty:
        vector_frames.append(frame[["image_path", "embedding"]])
    if path_index == 1 or path_index % 25 == 0 or path_index == len(embedding_paths):
        print(f"[all-available vectors {path_index}/{len(embedding_paths)}]", flush=True)
vectors = pd.concat(vector_frames, ignore_index=True).drop_duplicates("image_path", keep="last")
participant_images = participant_image_lookup.merge(vectors, on="image_path", how="left", validate="one_to_one")
if participant_images["embedding"].isna().any():
    raise RuntimeError("A selected matched participant image lacks its vector")
if set(participant_images["embedding"].map(lambda value: np.asarray(value).size)) != {expected_embedding_dim}:
    raise ValueError("Unexpected RETFound embedding dimension in matched cohort")

# Expand the image table by comparison because a White reference may legitimately
# participate in a different target group's separately matched cohort.
selection = primary_membership.merge(
    participant_images[["participant_id", "image_path", "selected_visit", "age", "racial_background", "embedding"]],
    on="participant_id", how="inner", validate="many_to_one",
).rename(columns={"target_group": "comparison_group", "selected_visit": "visit"})
if selection.duplicated(["comparison_group", "participant_id"]).any():
    raise ValueError("Matched selection is not unique by comparison and participant")

model_inventory_rows = []
prediction_rows = []
selection_rows = []
comparison_metric_rows = []
population_performance_rows = []

for group_index, comparison_group in enumerate(eligible_specific_groups):
    comparison = selection[selection["comparison_group"].astype(str) == comparison_group].copy()
    target = comparison[comparison["match_role"] == "target"].copy()
    white = comparison[comparison["match_role"] == "reference"].copy()
    if len(target) < minimum_all_available_n:
        raise ValueError(f"{comparison_group} retained only {len(target)} matched targets")
    target_prediction, target_fold, target_estimator, target_offset, target_splits = crossfit_population(
        target, all_available_seed + group_index * 100
    )
    white_prediction, white_fold, white_estimator, white_offset, white_splits = crossfit_population(
        white, all_available_seed + group_index * 100 + 1
    )
    target["oof_fold"] = target_fold
    white["oof_fold"] = white_fold
    target["population_oof_prediction"] = target_prediction
    white["population_oof_prediction"] = white_prediction

    slug = safe_group_name(comparison_group)
    bundles = {
        "target": {
            "model_name": f"CLSA_{comparison_group}_all_available_age_head",
            "estimator": target_estimator, "calibration_offset": target_offset,
            "embedding_dim": expected_embedding_dim, "ridge_alpha": ridge_alpha,
            "comparison_group": comparison_group, "model_role": "target",
            "racial_background": comparison_group, "n_training_participants": int(len(target)),
            "training_design": "all matched target participants; one image per participant",
        },
        "matched_white": {
            "model_name": f"CLSA_{comparison_group}_matched_White_age_head",
            "estimator": white_estimator, "calibration_offset": white_offset,
            "embedding_dim": expected_embedding_dim, "ridge_alpha": ridge_alpha,
            "comparison_group": comparison_group, "model_role": "matched_white",
            "racial_background": "White", "n_training_participants": int(len(white)),
            "training_design": "comparison-specific 1:2 matched White cohort; one image per participant",
        },
    }
    for role, bundle in bundles.items():
        filename = f"{slug}_{role}_age_head.joblib"
        local_model = Path("/local_disk0/tmp") / f"{uuid.uuid4().hex}_{filename}"
        joblib.dump(bundle, local_model)
        destination = all_available_model_root / filename
        publish_local_file(local_model, destination)
        local_model.unlink(missing_ok=True)
        model_inventory_rows.append({
            "comparison_group": comparison_group, "model_role": role,
            "model_path": str(destination), "n_training_participants": bundle["n_training_participants"],
        })

    # Cross-head evaluation without source-population leakage: use OOF prediction
    # on the source population and the full independent head on the other one.
    target_matrix = np.stack(target["embedding"].map(lambda value: np.asarray(value, dtype=np.float32)))
    white_matrix = np.stack(white["embedding"].map(lambda value: np.asarray(value, dtype=np.float32)))
    target_from_white = white_estimator.predict(target_matrix) + white_offset
    white_from_target = target_estimator.predict(white_matrix) + target_offset
    for population, frame, own_prediction, other_prediction in (
        ("target", target, target_prediction, target_from_white),
        ("matched_white", white, white_prediction, white_from_target),
    ):
        actual = frame["age"].to_numpy(float)
        source_predictions = {
            population: own_prediction,
            "matched_white" if population == "target" else "target": other_prediction,
        }
        for source_role, predicted in source_predictions.items():
            prediction_rows.append(pd.DataFrame({
                "comparison_group": comparison_group, "population_role": population,
                "source_model_role": source_role, "participant_id": frame["participant_id"].astype(str).to_numpy(),
                "image_path": frame["image_path"].astype(str).to_numpy(), "actual_age": actual,
                "predicted_retinal_age": predicted, "retinal_age_gap": predicted - actual,
                "absolute_error": np.abs(predicted - actual),
                "prediction_is_oof": source_role == population,
            }))
        selection_rows.append(frame.drop(columns=["embedding"]).assign(population_role=population))

    target_error_own = np.abs(target_prediction - target["age"].to_numpy(float))
    target_error_white = np.abs(target_from_white - target["age"].to_numpy(float))
    paired_difference = target_error_own - target_error_white
    rng = np.random.default_rng(all_available_seed + 7000 + group_index)
    boot = np.asarray([rng.choice(paired_difference, len(paired_difference), replace=True).mean() for _ in range(all_available_bootstraps)])
    sd_difference = float(np.std(paired_difference, ddof=1))
    mde80 = float((norm.ppf(0.975) + norm.ppf(0.80)) * sd_difference / np.sqrt(len(paired_difference)))
    p_value = min(1.0, max(1 / all_available_bootstraps, 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))))
    target_age = target["age"].to_numpy(float); white_age = white["age"].to_numpy(float)
    target_slope = float(np.polyfit(target_age, target_prediction, 1)[0])
    white_slope = float(np.polyfit(white_age, white_prediction, 1)[0])
    slope_boot = []
    for _ in range(all_available_bootstraps):
        target_index = rng.integers(0, len(target), len(target)); white_index = rng.integers(0, len(white), len(white))
        sampled_target_age = target_age[target_index]; sampled_white_age = white_age[white_index]
        if np.std(sampled_target_age) > 0 and np.std(sampled_white_age) > 0:
            slope_boot.append(
                np.polyfit(sampled_target_age, target_prediction[target_index], 1)[0]
                - np.polyfit(sampled_white_age, white_prediction[white_index], 1)[0]
            )
    slope_boot = np.asarray(slope_boot, dtype=float)
    slope_p = min(1.0, max(1 / len(slope_boot), 2 * min(np.mean(slope_boot <= 0), np.mean(slope_boot >= 0))))
    comparison_metric_rows.append({
        "comparison_group": comparison_group, "n_target_participants": int(len(target)),
        "n_matched_white_participants": int(len(white)), "target_cv_folds": target_splits,
        "white_cv_folds": white_splits, "target_head_minus_matched_white_head_mae": float(paired_difference.mean()),
        "ci_low": float(np.quantile(boot, 0.025)), "ci_high": float(np.quantile(boot, 0.975)),
        "p_value": float(p_value), "mde_80_percent_power_years": mde80,
        "target_minus_white_calibration_slope": target_slope - white_slope,
        "slope_ci_low": float(np.quantile(slope_boot, 0.025)), "slope_ci_high": float(np.quantile(slope_boot, 0.975)),
        "slope_p_value": float(slope_p),
        "slope_mde_80_percent_power": float((norm.ppf(0.975) + norm.ppf(0.80)) * np.std(slope_boot, ddof=1)),
    })
    for population_role, frame, predicted in (
        ("target", target, target_prediction),
        ("matched_white", white, white_prediction),
    ):
        actual = frame["age"].to_numpy(float)
        gap = predicted - actual
        slope, intercept = np.polyfit(actual, predicted, 1)
        population_performance_rows.append({
            "comparison_group": comparison_group, "population_role": population_role,
            "n_participants": int(len(frame)), "mae": float(np.mean(np.abs(gap))),
            "rmse": float(np.sqrt(np.mean(gap ** 2))), "mean_gap": float(np.mean(gap)),
            "calibration_slope": float(slope), "calibration_intercept": float(intercept),
            "correlation": float(np.corrcoef(actual, predicted)[0, 1]),
        })

model_inventory = pd.DataFrame(model_inventory_rows)
cross_head_predictions = pd.concat(prediction_rows, ignore_index=True)
expanded_selection = pd.concat(selection_rows, ignore_index=True)
comparison_metrics = pd.DataFrame(comparison_metric_rows)
population_performance = pd.DataFrame(population_performance_rows)
comparison_metrics["p_value_fdr"] = benjamini_hochberg(comparison_metrics["p_value"].to_numpy())
comparison_metrics["slope_p_value_fdr"] = benjamini_hochberg(comparison_metrics["slope_p_value"].to_numpy())
write_frame(model_inventory, all_available_statistics_root / "model_inventory.parquet")
write_frame(cross_head_predictions, private_root / "race_matched_all_available_cross_head_predictions_private.parquet")
write_frame(expanded_selection, private_root / "race_matched_all_available_image_selection_private.parquet")
write_frame(comparison_metrics, all_available_statistics_root / "target_vs_matched_white_head_mae.parquet")
write_frame(population_performance, all_available_statistics_root / "cross_fitted_population_performance.parquet")

display(expanded_selection.groupby(["comparison_group", "population_role"]).agg(participants=("participant_id", "nunique"), mean_age=("age", "mean")))
display(comparison_metrics.sort_values("p_value_fdr").round(4))
display(population_performance.round(4))


### Interpretation

Negative `target_head_minus_matched_white_head_mae` favors the target-group
head on that target group's participants. Because group-specific sample sizes
now differ, this analysis maximizes power but no longer isolates ethnicity from
training-sample size. Report it alongside—not instead of—the equal-N sensitivity
analysis. The global CLSA model remains the primary model.
